# Adding and splitting sectors

This notebook uses the packaged MARIO IOT test table to show the Excel workflow for extending a database with new sectors.

The screenshots below refer to the IOT workbook. SUT follows the same workflow, but the `Master` sheet uses `Activity` and `Commodity`, includes `Market share`, and does not expose `Add or Split`.

In [1]:
import mario

mario.set_log_verbosity("critical")

## Load the test table

We start from the packaged IOT test table so the workflow stays reproducible and easy to inspect.

In [2]:
db = mario.load_test("IOT")
db.get_index("Sector")

['Agriculture', 'Services', 'Industry']

## Understand the workbook

The add-sectors workbook is usually filled in two passes:

1. generate the empty workbook;
2. fill the `Master` sheet;
3. let MARIO create the missing inventory sheets;
4. fill the inventory sheets in Excel;
5. read the completed workbook and apply it.

The next sections explain what each sheet is for before running the workflow.

### The master sheet

![The add-sectors master sheet](../../_static/images/add_sector_master.png)

The `Master` sheet defines the high-level structure of the extension:

- `Region`: target database region. You can also use a region cluster name to replicate the same characterization in multiple regions.
- `Sector`: name of the new sector.
- `Inventory sheet`: name of the sheet where the inventory is described.
- `Quantity` and `Unit`: functional unit of the new sector's output.
- `Final consumption` and `Consumption category`: optional final demand attached to the new sector.
- `Parent Sector`: existing sector used as the reference when coefficients or percentage changes should inherit an existing structure.
- `Leave empty`: when set to `TRUE`, MARIO skips that inventory sheet.
- `Add or Split`: use `Add` for a fully new sector, or `Split` when the new sector must be extracted from a parent sector.

The same inventory sheet name can be reused across multiple `Master` rows when one characterization should be applied to multiple target regions.

### The inventory sheets

![The add-sectors inventory sheet](../../_static/images/add_sector_inventories.png)

Each inventory sheet describes the inputs used by the new sector:

- `Quantity`: technical coefficient per unit of output declared in `Master`.
- `Unit`: unit of that coefficient.
- `Input`: free user label; MARIO does not use it as a database key.
- `Item type`: database set to target, such as `Sector`, `Factor of production`, or `Satellite account`.
- `DB Item`: existing database label, or an item-cluster name when clusters are used.
- `DB Region`: source region of the input. This can be a database region or a region cluster.
- `Change type`: use `Update` for an absolute coefficient, or `Percentage` to scale the parent structure.
- `Source` and `Notes`: support columns for references and comments.

The `DB units` sheet in the workbook is a reference tab that helps you keep units consistent with the database.

### The cluster sheets

![The add-sectors cluster sheets](../../_static/images/add_sector_clusters.png)

Cluster sheets are optional, but they are useful whenever the same assumptions must be repeated:

- `Regions Clusters`: each column header is one cluster name, and each non-empty cell below it is one database region included in that cluster.
- `Sectors Clusters`: in IOT, this lets you group multiple existing sectors under one reusable label.
- `Commodities Clusters`: in SUT, the item-cluster sheet works in the same way for commodity inputs.

Region clusters can be referenced in `Master.Region` and in `Inventory.DB Region`. Item clusters can be referenced in `Inventory.DB Item`.

### Split-specific sheets

If at least one row in `Master` is marked as `Split`, MARIO also relies on `Total outputs`, `Trades`, `Exclusions`, and `Tolerances`.

For the moment, this is an IOT-only workflow. In practice, `Split` means that the new sector is extracted from an existing parent sector rather than added as a completely independent structure.

## Generate an empty add-sector template

In [3]:
db.get_add_sectors_excel(
    path="/Users/lorenzorinaldi/Documents/GitHub/MARIO/mario/test/supporting_files/add_sector_iot_template.xlsx",
    overwrite=True,
)

Fill the `Master` sheet in Excel first. Once the rows and inventory-sheet names are ready, read the workbook back with `get_inventories=True` so MARIO creates the missing inventory tabs inside the same file.

## Create inventory sheets from a filled `Master` sheet

In [4]:
db.read_add_sectors_excel(
    path="/Users/lorenzorinaldi/Documents/GitHub/MARIO/mario/test/supporting_files/add_sector_iot_master_filled.xlsx",
    get_inventories=True,
)

After filling the generated inventory tabs, read the completed workbook with `read_inventories=True`. MARIO will normalize the `Master` sheet, load clusters, and group the inventories by target sector.

## Read the completed workbook

In [5]:
db.read_add_sectors_excel(
    path="/Users/lorenzorinaldi/Documents/GitHub/MARIO/mario/test/supporting_files/add_sector_iot_inventories_filled.xlsx",
    read_inventories=True,
)

db.inventories.keys()

dict_keys(['New industry', 'New services'])

## Apply the completed workbook

Once the inventories are loaded, `db.add_sectors(...)` inserts the new sectors into the database. 

In [6]:
expanded_db = db.add_sectors(inplace=False)
expanded_db.get_index("Sector")

['Agriculture', 'Industry', 'New industry', 'New services', 'Services']

The updated coefficient and final-demand blocks now contain the new sectors.

In [7]:
expanded_db.Z

Region                              Reg1                             \
Level                             Sector                              
Item                         Agriculture      Industry New industry   
Region Level  Item                                                    
Reg1   Sector Agriculture   9.308777e+05  5.357865e+06          0.0   
              Industry      1.122412e+06  2.289312e+07          0.0   
              New industry  0.000000e+00  0.000000e+00          0.0   
              New services  0.000000e+00  0.000000e+00          0.0   
              Services      1.893611e+06  9.737629e+06          0.0   
Reg2   Sector Agriculture   4.168644e+02  2.596073e+03          0.0   
              Industry      6.891080e+03  1.335210e+05          0.0   
              New industry  0.000000e+00  0.000000e+00          0.0   
              New services  0.000000e+00  0.000000e+00          0.0   
              Services      1.590111e+03  1.555065e+04          0.0   

Region                                                         Reg2  \
Level                                                        Sector   
Item                       New services      Services   Agriculture   
Region Level  Item                                                    
Reg1   Sector Agriculture     15.435331  1.075287e+06   2363.627381   
              Industry       183.469920  1.065104e+07   1414.054877   
              New industry     0.000000  0.000000e+00      0.000000   
              New services     0.000000  0.000000e+00      0.000000   
              Services       439.223947  3.059810e+07   1302.054530   
Reg2   Sector Agriculture      0.012101  8.430140e+02   2963.307248   
              Industry       525.000000  6.555883e+04   9739.395361   
              New industry     0.000000  0.000000e+00      0.000000   
              New services     0.000000  0.000000e+00      0.000000   
              Services         0.551754  3.843739e+04  16684.555239   

Region                                                               \
Level                                                                 
Item                             Industry New industry New services   
Region Level  Item                                                    
Reg1   Sector Agriculture    50638.344971          0.0          0.0   
              Industry      134992.066256          0.0          0.0   
              New industry       0.000000          0.0          0.0   
              New services       0.000000          0.0          0.0   
              Services       27830.640844          0.0          0.0   
Reg2   Sector Agriculture    30653.531705          0.0          0.0   
              Industry      287474.038293          0.0          0.0   
              New industry       0.000000          0.0          0.0   
              New services       0.000000          0.0          0.0   
              Services      312544.259102          0.0          0.0   

Region                                     
Level                                      
Item                             Services  
Region Level  Item                         
Reg1   Sector Agriculture    10343.097378  
              Industry       30370.678292  
              New industry       0.000000  
              New services       0.000000  
              Services       58532.069101  
Reg2   Sector Agriculture     7962.553289  
              Industry      161145.344747  
              New industry       0.000000  
              New services       0.000000  
              Services      722171.520941

In [8]:
expanded_db.Y

Region                                     Reg1                 Reg2
Level                      Consumption category Consumption category
Item                               Final demand         Final demand
Region Level  Item                                                  
Reg1   Sector Agriculture          2.533459e+06         1.145534e+04
              Industry             1.816087e+07         1.217507e+05
              New industry         0.000000e+00         0.000000e+00
              New services         1.500000e+03         0.000000e+00
              Services             6.212571e+07         5.286975e+04
Reg2   Sector Agriculture          5.077805e+03         2.935670e+04
              Industry             2.316510e+05         2.998989e+05
              New industry         0.000000e+00         0.000000e+00
              New services         0.000000e+00         0.000000e+00
              Services             5.695455e+04         1.255005e+06

## Split the new sector with CVXLab

The split workflow reuses the same inventories already loaded above, but asks MARIO to hand the split scenario to a CVXLab model.

The next cell is still useful as the reference call for the intended workflow: once the CVXLab split model is aligned with the database layout, it should return a new `expanded_db_split` scenario.

At the moment, depending on the CVXLab model version, this call can fail before optimization starts. The current incompatibility is in the CVXLab-side handling of factor rows inside `VA`, not in the `db.add_sectors(..., split=True)` call itself.

In [9]:
expanded_db_split = db.add_sectors(
    inplace=False, 
    split=True,
    cvxlab_path="/Users/lorenzorinaldi/Documents/GitHub/MARIO/mario/test/supporting_files/cvxlab",
    )


INFO | Model | Model instance generation...
INFO | Model.core.database | Generating new sets excel file 'sets.xlsx'.
INFO | Model | Model instance generation... DONE (0.05 seconds)
INFO | Model | Loading sets and variables coordinates...
INFO | Model | Loading sets and variables coordinates... DONE (0.02 seconds)
INFO | Model | Generation of blank data structures...
INFO | Model | Creating new blank SQLite database 'database.db'.
INFO | Model | Generating new blank input data directory and related file/s.
INFO | Model | Generation of blank data structures... DONE (0.26 seconds)
INFO | Model | Loading exogenous data into SQLite database 'database.db' and initializing problems.
INFO | Model | Loading input data to SQLite database...
ERROR | Model.core.sql_manager | SQLite table 'VA' | action 'update' | length mismatch (existing table: 2, passed data: 6)
INFO | Model | Loading input data to SQLite database... FAILED (0.06 seconds)


OperationalError: SQLite table 'VA' | action 'update' | length mismatch (existing table: 2, passed data: 6)

In [10]:
expanded_db_split.Z

NameError: name 'expanded_db_split' is not defined